# Día 11 · Perfil Inteligente de Riesgo

Estandariza severidad y probabilidad, calcula el PIRD y genera el perfil ejecutivo. Requiere clave de OpenAI, pero no GPU.


In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-11-perfil-inteligente
!pip -q install openai pandas numpy scikit-learn


## 1. Clave y archivos privados
Sube `resultados_clasificador_calibrado_dia_05.zip` y `resultados_clustering_semantico_dia_09.zip`.


In [ ]:
import getpass,os,zipfile,subprocess,json,shutil
from pathlib import Path
from google.colab import files
os.environ['OPENAI_API_KEY']=getpass.getpass('Ingrese OPENAI_API_KEY: ')
uploaded=files.upload();private=Path('/content/day11_private');private.mkdir(exist_ok=True)
for name in uploaded:
    if 'clasificador' in name:
        with zipfile.ZipFile(name) as z:z.extract('surveillance_candidates_calibrated.csv',private)
    elif 'clustering' in name:
        with zipfile.ZipFile(name) as z:
            z.extract('cluster_assignments_private.csv',private);z.extract('signal_embeddings_bge_m3.npy',private)


## 2. Reconstruir timeline corregido


In [ ]:
timeline_out=Path('/content/day11_timeline')
!python -m src.risk.build_risk_timeline --items-csv /content/day11_private/surveillance_candidates_calibrated.csv --assignments-csv /content/day11_private/cluster_assignments_private.csv --embeddings-npy /content/day11_private/signal_embeddings_bge_m3.npy --output-dir /content/day11_timeline


## 3. Evaluar exposición y generar perfil
La ejecución realiza aproximadamente 65 llamadas en lotes de 10 y conserva checkpoint.


In [ ]:
out=Path('/content/resultados_perfil_inteligente_dia_11')
!python -m src.risk.build_intelligent_risk_profile --items-csv /content/day11_private/surveillance_candidates_calibrated.csv --timeline-csv /content/day11_timeline/risk_timeline_private.csv --output-dir /content/resultados_perfil_inteligente_dia_11 --batch-size 10


## 4. Resultados y descarga privada


In [ ]:
print(json.dumps(json.loads((out/'intelligent_risk_profile.json').read_text()),ensure_ascii=False,indent=2))
archive=shutil.make_archive('/content/resultados_perfil_inteligente_dia_11','zip',out)
files.download(archive)
